In [66]:
import torch
import numpy as np
from transformers import AutoModelForMaskedLM, AutoTokenizer
import torch.nn.functional as F



In [67]:
class MaskedLMRanker:

  def __init__(self, model_name: str):
    self.tokenizer = AutoTokenizer.from_pretrained(model_name)
    self.model = AutoModelForMaskedLM.from_pretrained(model_name)
    self.model.eval()
    self.mask_token = self.tokenizer.mask_token
    self.model_name = model_name



  # calculating the probabilty of target_word appearing the context. high = good
  def get_exemplification_score(self, sentence, target_word):



    #IDENTIFY WHERE THE WORD IS AND HANDLE MULTI-TOKENIZATION

    #search for the word in the wokenized sequence to ensure alignments

    candidate_forms = set([target_word, target_word.lower(), target_word.capitalize(),
                           " " + target_word,
                           " " + target_word.lower(),
                           " "  +target_word.capitalize()])

    target_variants = []
    seen_id_tuples = set()
    for form in candidate_forms:
      toks = self.tokenizer.tokenize(form)
      ids = tuple(self.tokenizer.convert_tokens_to_ids(toks))
      if ids not in seen_id_tuples:
        seen_id_tuples.add(ids)
        target_variants.append((toks,list(ids)))


    # target_tokens = self.tokenizer.tokenize(target_word)
    # target_ids = self.tokenizer.convert_tokens_to_ids(target_tokens)
    # num_target_tokens = len(target_ids)




    #tokenize whole sentence
    inputs = self.tokenizer(sentence, return_tensors = "pt")
    input_ids = inputs["input_ids"][0]



    #FINDING ALL OCCURENCES OF THE TARGET WORD IN TOKENIZED SENTENCE
    occurrences = []
    for (toks, t_ids) in target_variants:
      n = len(t_ids)
      for i in range(len(input_ids) - n+ 1):
        if input_ids[i : i + n].tolist() == t_ids:
          occurrences.append((list(range(i, i + n)), t_ids))

    if not occurrences:
      return {"status" : "failure",
              "reason" : "target_word not found in sentence"}


    #CREATING MASKED VERSION OF THE SENTENCE - mask all occurences of target word simultaenously
    masked_input_ids = input_ids.clone()
    for (idx_list, _) in occurrences:
      for idx in idx_list:
        masked_input_ids[idx] = self.tokenizer.mask_token_id

    #MODEL INFERENCE
    with torch.no_grad():
      outputs = self.model(masked_input_ids.unsqueeze(0),
                           attention_mask = inputs["attention_mask"])
      logits = outputs.logits[0]        #shapes:[seq_len, vocab_size]



    #PROBABILITY - log_softmax for numerical stability
    log_probs = F.log_softmax(logits, dim=-1)
    total_log_prob =0.0
    total_tokens_evaluated = 0

    for (idx_list, t_ids) in occurrences:
      for i, token_idx in enumerate(idx_list):
        target_token_id = t_ids[i]
        token_log_prob = log_probs[token_idx, target_token_id].item()
        total_log_prob += token_log_prob
        total_tokens_evaluated += 1


    score = total_log_prob/total_tokens_evaluated   #normalised score - mean log probability across all occurences, log_prob is better for ranking
    return{"status": "SUCCESS",
           "score": np.exp(score),
           "log_prob": score,
           "tokens": [t for (toks, _) in occurrences[:1] for t in toks]}


    def rerank(self, candidates: list, target_word: str) -> list:
    #Score all candidate sentences and return them sorted best-first.
      results = []
      for sent in candidates:
        res = self.get_exemplification_score(sent, target_word)
        if res["status"] == "SUCCESS":
          results.append({"sentence": sent, **res})
        else:
          print(f"[WARN] Could not score: '{sent[:60]}'")
      results.sort(key=lambda x: x["score"], reverse=True)
      return results






In [68]:
hindi_ranker = MaskedLMRanker("google/muril-base-cased")


sentences = ["वह बाज़ार से फल खरीदने गया था।",
             "मैंने फल देखा।",
             "उसने मुझे फल दिया।"]

for sent in sentences:
  res = hindi_ranker.get_exemplification_score(sent, "फल")
  print(f"Sentence: {sent}\nScore: {res['score']:.4f}\n")


Loading weights:   0%|          | 0/204 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie bert.embeddings.word_embeddings.weight to cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie cls.predictions.bias to cls.predictions.decoder.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
BertForMaskedLM LOAD REPORT from: google/muril-base-cased
Key                          | Status     |  | 
-----------------------------+------------+--+-
cls.seq_relationship.bias    | UNEXPECTED |  | 
cls.seq_relationship.weight  | UNEXPECTED |  | 
bert.pooler.dense.bias       | UNEXPECTED |  | 
bert.embeddings.position_ids | UNEXPECTED |  | 
bert.pooler.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored w

Sentence: वह बाज़ार से फल खरीदने गया था।
Score: 0.0071

Sentence: मैंने फल देखा।
Score: 0.0000

Sentence: उसने मुझे फल दिया।
Score: 0.0003



In [65]:
english_ranker = MaskedLMRanker("roberta-large")
target = "bank"
candidates = [
    "I went to the bank to deposit my paycheck.",             # High context (deposit, paycheck)
    "The fisherman sat on the river bank.",                   # High context (fisherman, river)
    "I will meet you near the bank.",                         # Low context (could be any building)
    "He had to bank on his friend's help to finish the job." # Different meaning (verb)
]

ranked_results = []
for sent in candidates:
    prob = english_ranker.get_exemplification_score(sent, target)
    if prob["status"] == "SUCCESS":
        ranked_results.append((sent, prob))
    else:
      print(f"Skippked: {sent} -> {prob['reason']}")

# Sort by probability (highest first)
ranked_results.sort(key=lambda x: x[1]['score'], reverse=True)

print(f"\nResults for target word: '{target}'")
for i, (s, p) in enumerate(ranked_results):
    print(f"{i+1}. {s} [Score: {p['score']:.4f}] ")

Loading weights:   0%|          | 0/394 [00:00<?, ?it/s]

RobertaForMaskedLM LOAD REPORT from: roberta-large
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Results for target word: 'bank'
1. I went to the bank to deposit my paycheck. [Score: 0.9938] 
2. The fisherman sat on the river bank. [Score: 0.3283] 
3. I will meet you near the bank. [Score: 0.0060] 
4. He had to bank on his friend's help to finish the job. [Score: 0.0008] 
